<a href="https://colab.research.google.com/github/Ceciluh/proyecto_TFOD_YOLO/blob/main/YOLO_project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 65.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import os

model = YOLO("yolov8s.pt")

#Trainning
results = model.train(
    data=r"/content/drive/MyDrive/YOLO_project/config.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    name="yolov8n_detection",
    project="runs/detect",
    device=0,
    patience=50,
    save=True,
    plots=True
)

Ultralytics 8.3.211 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/YOLO_project/config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_detection3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspe

FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/yolov8n_detection/weights/best.pt'

In [ ]:
import time

metrics=model.val()
print("============ Metricas ==============")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

#Velocidad de inferencia
print("============ Velocidad de inferencia ==============")

best_model_path="runs/detect/yolov8n_detection3/weights/best.pt"
model=YOLO(best_model_path)

test_image_path="/content/drive/MyDrive/YOLO_project/data/images/val/office_objects_59.jpg"


for i in range(10):
    model(test_image_path, verbose=False)

inference_times = []

for i in range(100):
    start_time = time.time()
    results = model(test_image_path, verbose=False)
    end_time = time.time()

    inference_times.append((end_time - start_time) * 1000)

avg_time=sum(inference_times) / len(inference_times)
min_time=min(inference_times)
max_time=max(inference_times)
fps = 1000 / avg_time

print(f"Tiempo promedio: {avg_time:.4f} ms")
print(f"Tiempo minimo: {min_time:.4f} ms")
print(f"Tiempo maximo: {max_time:.4f} ms")
print(f"FPS: {fps:.4f}")

# Tamano del modelo
print("=========== Tamaño del modelo ===============")
model_size_mb=os.path.getsize(best_model_path) / (1024 * 1024)
print(f"Tamaño del modelo: {model_size_mb:.4f} mb")
print("=============================================")

Ultralytics 8.3.211 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.4±0.2 ms, read: 84.9±77.1 MB/s, size: 192.1 KB)
val: Scanning /content/drive/MyDrive/YOLO_project/data/labels/val.cache... 103 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 103/103 129.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.6it/s 4.3s
                   all        103        151      0.944       0.82      0.897       0.72
            Calculator         15         15      0.919      0.758      0.861      0.719
                  Lamp         13         16          1      0.929      0.955      0.734
              Keyboard         27         28      0.907       0.75      0.845      0.671
                Laptop         18         23      0.944      0.957      0.942      0.796
               Monitor         25         32      0.957      0.688       0.79      0.694
                 Mous

In [ ]:
#Exportar a ONNX
onnx_path = best_model.export(format="onnx", dynamic=True, simplify=True)
onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX: {onnx_size_mb:.2f} MB")